In [7]:
import argparse
import json
import logging

import pandas as pd
from article_utils.util import dump_jsonl
from doccano_client import DoccanoClient

from doccano.util import (
    get_example,
    get_project_by_name,
    list_annotations,
    list_comments,
    list_examples,
    list_members,
    list_users,
    add_label_to_example
)


with open('.doccano_users.json', 'r') as fh:
    passwords = json.load(fh)

# instantiate a client and log in to a Doccano instance
client = DoccanoClient('https://doccano.bcgsc.ca')
client.login(username='admin', password=passwords['admin'])

proj = get_project_by_name(client,"Time Split Analysis (2026.06.09)")
members = list_members(client, proj['id'])
admin_member = [u for u in members if u['username'] == 'admin'][0]
print(members)
proj

[{'id': 433, 'user': 1, 'role': 1, 'username': 'admin', 'rolename': 'project_admin'}]


{'id': 104,
 'name': 'Time Split Analysis (2026.06.09)',
 'description': 'Evaluating results from retreival from Qwen3 8B and MedCPT fine-tuned as well as the subsequent claim verification results for Gemma4, GPT 5.2, and our FT Llama 3.2 model. We will evaluate the validity of the input pairs as well as the validity of the model responses.',
 'guideline': '',
 'project_type': 'DocumentClassification',
 'created_at': '2026-06-10T19:59:33.461707Z',
 'updated_at': '2026-06-10T19:59:33.461722Z',
 'random_order': False,
 'author': 'admin',
 'collaborative_annotation': False,
 'single_class_classification': False,
 'allow_member_to_create_label_type': True,
 'is_text_project': True,
 'tags': [{'id': 1, 'project': 104, 'text': 'retrieval_ok_qwen'},
  {'id': 2, 'project': 104, 'text': 'retrieval_ok_medcpt'}],
 'resourcetype': 'TextClassificationProject'}

In [8]:
# for every 6th example add the "main" label

def get_labelname2id_mapping(client, project_type: str, project_id: int) -> dict[str, str]:

    if project_type == 'SequenceLabeling':

        span_types = {}
        for span_type in client._base_repository.get(f'projects/{project_id}/span-types').json():
            span_types[span_type['labelname']] = span_type['id']
        return span_type
    elif project_type == 'DocumentClassification':
        tag_types = {}
        for tag in client._base_repository.get(f'projects/{project_id}/category-types').json():
            tag_types[tag['text']] = tag['id']
        return tag_types
    else:
        raise NotImplementedError(f'not implmented yet for project type: {project_type}')

examples = list_examples(client, proj['id'])
mapping = get_labelname2id_mapping(client, proj['project_type'], proj['id'])
mapping

{'retrieval_ok_qwen': 2604,
 'retrieval_ok_medcpt': 2605,
 'repurposed_eid': 2606,
 'new_label_nei': 2607,
 'new_label_revisions': 2608,
 'need_second_opinion': 2609,
 'added_to_hypothesis': 2610,
 'bad_accepted_revision': 2611,
 'new_label_supports': 2612,
 'model_revision_pointless': 2613,
 'requires_supplement': 2614,
 'super_minor_revision': 2615,
 'wrong_or_bad_source': 2616,
 'bad_linked_term': 2617,
 'retrieval_failed_qwen': 2618,
 'retrieval_failed_medcpt': 2619,
 'civic_entry_ok': 2620,
 'model_revision_ok': 2621,
 'model_revision_incorrect': 2622,
 'possible_hallucination': 2623,
 'requires_image': 2624,
 'no_actual_revision': 2625,
 'exp_revision_approx_eq_claim': 2626,
 'not_in_main_text': 2627,
 'major_revision': 2628,
 'main': 2629}

In [12]:
import requests
for i in range(0, len(examples), 6):
    ex = examples[i]
    try:
        add_label_to_example(client, proj['id'], ex['id'], mapping['main'], admin_member['id'])
        print('ok')
    except requests.HTTPError as err:
        print(err)

500 Server Error: Internal Server Error for url: https://doccano.bcgsc.ca/v1/projects/104/examples/124286/categories
500 Server Error: Internal Server Error for url: https://doccano.bcgsc.ca/v1/projects/104/examples/124292/categories
500 Server Error: Internal Server Error for url: https://doccano.bcgsc.ca/v1/projects/104/examples/124298/categories
500 Server Error: Internal Server Error for url: https://doccano.bcgsc.ca/v1/projects/104/examples/124304/categories
500 Server Error: Internal Server Error for url: https://doccano.bcgsc.ca/v1/projects/104/examples/124310/categories
500 Server Error: Internal Server Error for url: https://doccano.bcgsc.ca/v1/projects/104/examples/124316/categories
500 Server Error: Internal Server Error for url: https://doccano.bcgsc.ca/v1/projects/104/examples/124322/categories
500 Server Error: Internal Server Error for url: https://doccano.bcgsc.ca/v1/projects/104/examples/124328/categories
500 Server Error: Internal Server Error for url: https://doccano